In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
len(docs)

9

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [6]:
len(splitted_docs)

26

In [7]:
embeddings = OllamaEmbeddings(
    model = "qwen3-embedding:8b"
)

In [8]:
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding= embeddings
)

In [19]:
same_record = vector_store.similarity_search("Patient name")

In [20]:
same_record[0].page_content

'delivery may be delayed due to unforeseen circumstances. Inconvenience is regretted .ŸCertain tests may require further testing at additional cost \nfor derivation of exact value. Kindly submit request within 72 hours post reporting .ŸTest results may show interlaboratory variations .ŸThe \nCourts/Forum at Delhi shall have exclusive jurisdiction in all disputes /claims concerning the test (s) & or results of test (s).ŸTest results are not valid \nfor medico legal purposes .ŸThis is computer generated medical diagnostic report that has been validated by Authorized Medical \nPractitioner/Doctor.ŸThe report does not need physical signature. \n(#) Sample drawn from outside source.\nIf Test results are alarming or unexpected, client is advised to contact the Customer Care immediately for possible remedial action. \nTel: +91-11-49885050,Fax: - +91-11-2788-2134, E-mail: lalpathlabs@lalpathlabs.com'

## agent = tools, llm, prompt

In [33]:
@tool
def retriever_tool(query:str):
    """
    This tool can help ypu to retrive the relevant data of the PDF Documents, and these PDF documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query = query, k = 4)
    print(len(docs), docs[0].page_content)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context

In [34]:
retriever_tool.invoke("Patient Name")

Tool Called:  Patient Name
4 delivery may be delayed due to unforeseen circumstances. Inconvenience is regretted .ŸCertain tests may require further testing at additional cost 
for derivation of exact value. Kindly submit request within 72 hours post reporting .ŸTest results may show interlaboratory variations .ŸThe 
Courts/Forum at Delhi shall have exclusive jurisdiction in all disputes /claims concerning the test (s) & or results of test (s).ŸTest results are not valid 
for medico legal purposes .ŸThis is computer generated medical diagnostic report that has been validated by Authorized Medical 
Practitioner/Doctor.ŸThe report does not need physical signature. 
(#) Sample drawn from outside source.
If Test results are alarming or unexpected, client is advised to contact the Customer Care immediately for possible remedial action. 
Tel: +91-11-49885050,Fax: - +91-11-2788-2134, E-mail: lalpathlabs@lalpathlabs.com


'Tel: +91-11-49885050,Fax: - +91-11-2788-2134, E-mail: lalpathlabs@lalpathlabs.com \nNational Reference lab, Delhi, a CAP (7171001) Accredited, ISO 9001:2015 (FS60411) & ISO 27001:2013 (616691) Certified laboratory.\n*474764803*\n.\nPage 9 of 9\n\n'

In [35]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b"
)

In [36]:
System_Prompt = """
You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge. 
"""

In [37]:
agent = create_agent(
    model = llm,
    tools = [retriever_tool],
    system_prompt= System_Prompt
)

In [38]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  patient name doctor names medical report
4 delivery may be delayed due to unforeseen circumstances. Inconvenience is regretted .ŸCertain tests may require further testing at additional cost 
for derivation of exact value. Kindly submit request within 72 hours post reporting .ŸTest results may show interlaboratory variations .ŸThe 
Courts/Forum at Delhi shall have exclusive jurisdiction in all disputes /claims concerning the test (s) & or results of test (s).ŸTest results are not valid 
for medico legal purposes .ŸThis is computer generated medical diagnostic report that has been validated by Authorized Medical 
Practitioner/Doctor.ŸThe report does not need physical signature. 
(#) Sample drawn from outside source.
If Test results are alarming or unexpected, client is advised to contact the Customer Care immediately for possible remedial action. 
Tel: +91-11-49885050,Fax: - +91-11-2788-2134, E-mail: lalpathlabs@lalpathlabs.com


In [39]:
result = response["messages"][-1].content

In [40]:
print(result)

**Patient**

- **Ms. Nikita Chudhary**

**Doctors mentioned in the report**

| Doctor | Title / Role (as given) |
|--------|-------------------------|
| **Dr Nitin Nahar** | Reported (likely the attending physician) |
| **Dr Beena Chandrasekhar, PhD** | Technical Director – Flow Cytometry |
| **Dr Nagarjun Sai Jaine, MD** | Consultant – Hemato‑Oncology & Flow Cytometry |
| **Dr Sunanda** | Senior Consultant Pathologist |

These are the names that appear in the retrieved medical report.
